### 1. Ingestão dos CSVs para Bronze

Este projeto compara nove séries de TV com base em dados de episódios e avaliações do IMDb. O pipeline foi desenvolvido no Databricks com arquitetura medalhão. 
Os CSVs são preservados na camada Bronze, tratados na Silver e organizados na Gold para responder às perguntas de negócio.
Estes CSVs foram exportados do Kaggle, de alguns datasets criados por apenas um usuário: https://www.kaggle.com/bcruise/datasets


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType

catalogo = "mvp_engdados"
root = "/Volumes/mvp_engdados/bronze/datafiles/"

display(dbutils.fs.ls(root))

path,name,size,modificationTime
dbfs:/Volumes/mvp_engdados/bronze/datafiles/big_bang_theory_episodes.csv,big_bang_theory_episodes.csv,51760,1790218084000
dbfs:/Volumes/mvp_engdados/bronze/datafiles/big_bang_theory_imdb.csv,big_bang_theory_imdb.csv,60221,1790218084000
dbfs:/Volumes/mvp_engdados/bronze/datafiles/breaking_bad_episodes.csv,breaking_bad_episodes.csv,4557,1790218083000
dbfs:/Volumes/mvp_engdados/bronze/datafiles/breaking_bad_imdb.csv,breaking_bad_imdb.csv,11740,1790218083000
dbfs:/Volumes/mvp_engdados/bronze/datafiles/brooklyn99_episodes.csv,brooklyn99_episodes.csv,12762,1790218083000
dbfs:/Volumes/mvp_engdados/bronze/datafiles/brooklyn99_imdb.csv,brooklyn99_imdb.csv,30334,1790218083000
dbfs:/Volumes/mvp_engdados/bronze/datafiles/friends_episodes.csv,friends_episodes.csv,27179,1790218083000
dbfs:/Volumes/mvp_engdados/bronze/datafiles/friends_imdb.csv,friends_imdb.csv,61047,1790218083000
dbfs:/Volumes/mvp_engdados/bronze/datafiles/game_of_thrones_episodes.csv,game_of_thrones_episodes.csv,6334,1790218083000
dbfs:/Volumes/mvp_engdados/bronze/datafiles/game_of_thrones_imdb.csv,game_of_thrones_imdb.csv,16518,1790218083000


Nesta etapa, os arquivos CSV do volume são lidos como texto para preservar os valores originais. O identificador da série é extraído do nome do arquivo, e cada registro recebe o caminho de origem e o horário de ingestão. As tabelas Bronze são gravadas em Delta sem aplicar as correções de negócio encontradas na análise.

In [0]:
def listar_csvs(caminho):
    arquivos = []

    for item in dbutils.fs.ls(caminho):
        if item.isDir():
            arquivos.extend(listar_csvs(item.path))
        elif item.name.lower().endswith(".csv"):
            arquivos.append(item.path)

    return sorted(arquivos)


csv_paths = listar_csvs(root)

mapa_series = {
    "brooklyn99": "brooklyn_99",
    "himym": "how_i_met_your_mother"
}

episodios_dfs = []
imdb_dfs = []

for path in csv_paths:
    nome_arquivo = path.rsplit("/", 1)[-1].lower()

    if nome_arquivo.endswith("_episodes.csv"):
        prefixo = nome_arquivo.removesuffix("_episodes.csv")
        destino = episodios_dfs
    elif nome_arquivo.endswith("_imdb.csv"):
        prefixo = nome_arquivo.removesuffix("_imdb.csv")
        destino = imdb_dfs
    else:
        continue

    series_id = mapa_series.get(prefixo, prefixo)

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("mode", "PERMISSIVE")
        .csv(path)
        .withColumn("series_id", F.lit(series_id))
        .withColumn("_source_file", F.lit(path))
        .withColumn("_ingested_at", F.current_timestamp())
    )

    destino.append(df)


if not episodios_dfs:
    raise ValueError(f"Nenhum CSV de episódios encontrado em {root}")

if not imdb_dfs:
    raise ValueError(f"Nenhum CSV de IMDb encontrado em {root}")


def unir_dataframes(dataframes):
    resultado = dataframes[0]

    for df in dataframes[1:]:
        resultado = resultado.unionByName(
            df,
            allowMissingColumns=True
        )

    return resultado


episodios_bronze = unir_dataframes(episodios_dfs)
imdb_bronze = unir_dataframes(imdb_dfs)

display(
    episodios_bronze
    .groupBy("series_id")
    .count()
    .orderBy("series_id")
)

display(
    imdb_bronze
    .groupBy("series_id")
    .count()
    .orderBy("series_id")
)


(
    episodios_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_engdados.bronze.episodes_raw")
)

(
    imdb_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_engdados.bronze.imdb_raw")
)

series_id,count
big_bang_theory,279
breaking_bad,62
brooklyn_99,153
friends,236
game_of_thrones,73
how_i_met_your_mother,208
lost,121
the_office,201
the_walking_dead,177


series_id,count
big_bang_theory,280
breaking_bad,62
brooklyn_99,153
friends,235
game_of_thrones,73
how_i_met_your_mother,208
lost,119
the_office,188
the_walking_dead,177


### 2. Perfil de qualidade da Bronze

Antes de transformar os dados, é feito nesse próximo passo um perfil das tabelas brutas. As consultas abaixo medem volumes por série, valores nulos ou vazios, duplicidades candidatas e limites de notas, votos e audiência. 
Os achados devem ser explicados como observações dos dados. Valores ausentes, por exemplo, não devem ser preenchidos sem uma regra ou fonte que justifique o preenchimento.

In [0]:
episodios = spark.table("mvp_engdados.bronze.episodes_raw")
imdb = spark.table("mvp_engdados.bronze.imdb_raw")

print("Total de episódios:", episodios.count())
print("Total de registros IMDb:", imdb.count())

display(
    episodios
    .groupBy("series_id")
    .count()
    .orderBy("series_id")
)

display(
    imdb
    .groupBy("series_id")
    .count()
    .orderBy("series_id")
)


def mostrar_nulos_ou_vazios(df, nome_tabela):
    colunas_texto = [
        campo.name
        for campo in df.schema.fields
        if isinstance(campo.dataType, StringType)
    ]

    expressoes = [
        F.sum(
            F.when(
                F.col(coluna).isNull()
                | (F.trim(F.col(coluna)) == ""),
                1
            ).otherwise(0)
        ).alias(coluna)
        for coluna in colunas_texto
    ]

    print(f"Valores nulos ou vazios — {nome_tabela}")
    display(df.agg(*expressoes))


mostrar_nulos_ou_vazios(episodios, "episodes_raw")
mostrar_nulos_ou_vazios(imdb, "imdb_raw")


display(
    episodios
    .groupBy("series_id", "season", "episode_num_in_season")
    .count()
    .filter(F.col("count") > 1)
    .orderBy("series_id", "season", "episode_num_in_season")
)

display(
    imdb
    .groupBy("series_id", "season", "episode_num")
    .count()
    .filter(F.col("count") > 1)
    .orderBy("series_id", "season", "episode_num")
)


# Datas IMDb como chegaram na Bronze
display(
    imdb
    .select("original_air_date")
    .distinct()
    .orderBy("original_air_date")
    .limit(50)
)


# Conversões temporárias para investigar valores e faixas
display(
    imdb
    .select(
        "series_id",
        F.expr(
            "try_cast(nullif(trim(imdb_rating), '') AS DOUBLE)"
        ).alias("imdb_rating_num"),
        F.expr(
            "try_cast(nullif(trim(total_votes), '') AS BIGINT)"
        ).alias("total_votes_num")
    )
    .groupBy("series_id")
    .agg(
        F.min("imdb_rating_num").alias("menor_nota"),
        F.max("imdb_rating_num").alias("maior_nota"),
        F.avg("imdb_rating_num").alias("nota_media"),
        F.min("total_votes_num").alias("menor_numero_votos"),
        F.max("total_votes_num").alias("maior_numero_votos")
    )
    .orderBy("series_id")
)

display(
    episodios
    .select(
        "series_id",
        F.expr(
            "try_cast(nullif(trim(us_viewers), '') AS DOUBLE)"
        ).alias("us_viewers_num")
    )
    .groupBy("series_id")
    .agg(
        F.min("us_viewers_num").alias("menor_audiencia"),
        F.max("us_viewers_num").alias("maior_audiencia"),
        F.avg("us_viewers_num").alias("audiencia_media"),
        F.count("us_viewers_num").alias("episodios_com_audiencia")
    )
    .orderBy("series_id")
)

Total de episódios: 1510
Total de registros IMDb: 1495


series_id,count
big_bang_theory,279
breaking_bad,62
brooklyn_99,153
friends,236
game_of_thrones,73
how_i_met_your_mother,208
lost,121
the_office,201
the_walking_dead,177


series_id,count
big_bang_theory,280
breaking_bad,62
brooklyn_99,153
friends,235
game_of_thrones,73
how_i_met_your_mother,208
lost,119
the_office,188
the_walking_dead,177


Valores nulos ou vazios — episodes_raw


season,episode_num_in_season,episode_num_overall,title,directed_by,written_by,original_air_date,prod_code,us_viewers,series_id,_source_file,featured_character
0,0,0,0,0,0,0,433,5,0,0,1389


...

series_id,season,episode_num_in_season,count
lost,6,12,2


series_id,season,episode_num,count


original_air_date
1 Apr. 2009
1 Apr. 2012
1 Apr. 2018
1 Dec. 2004
1 Dec. 2016
1 Feb. 1996
1 Feb. 2001
1 Feb. 2010
1 Feb. 2018
1 Jan. 2017


series_id,menor_nota,maior_nota,nota_media,menor_numero_votos,maior_numero_votos
big_bang_theory,6.7,9.5,7.813214285714288,1888,8791
breaking_bad,7.9,10.0,9.02741935483871,16545,150341
brooklyn_99,4.8,9.6,8.127450980392155,1178,6225
friends,7.1,9.7,8.422553191489362,3427,14033
game_of_thrones,4.1,9.9,8.753424657534246,25968,242548
how_i_met_your_mother,5.5,9.5,8.141346153846156,2678,18191
lost,7.2,9.7,8.547899159663869,843,12557
the_office,6.4,9.8,8.219148936170212,2724,15245
the_walking_dead,4.1,9.6,7.915819209039551,4325,42427


series_id,menor_audiencia,maior_audiencia,audiencia_media,episodios_com_audiencia
big_bang_theory,7340000.0,2.044E7,1.4646308243727598E7,279
breaking_bad,970000.0,1.028E7,2324385.964912281,57
brooklyn_99,1310000.0,1.507E7,2620718.954248366,153
friends,1.565E7,5.29E7,2.539080508474576E7,236
game_of_thrones,2200000.0,1.361E7,6447808.219178082,73
how_i_met_your_mother,6410000.0,1.313E7,9044182.692307692,208
lost,8289999.999999999,2.347E7,1.458099173553719E7,121
the_office,3250000.0,2.291E7,7358855.721393035,201
the_walking_dead,1190000.0,1.729E7,8137966.101694915,177


A duplicidade detectada em Lost, temporada 6, episódio 12, foi investigada antes de qualquer alteração. A consulta abaixo exibe o episódio e o contexto da sequência. A evidência apontou que LA X - Part 2, com número geral 105, deveria ser o episódio 2 da temporada; o episódio 12 era Everybody Loves Hugo. Por isso, o ajuste será registrado e aplicado na Silver, preservando na Bronze o valor original.

In [0]:
display(
    episodios
    .filter(
        (F.col("series_id") == "lost")
        & (F.col("season") == "6")
        & (F.col("episode_num_in_season") == "12")
    )
    .select(
        "series_id",
        "season",
        "episode_num_in_season",
        "episode_num_overall",
        "title",
        "original_air_date",
        "prod_code",
        "us_viewers",
        "_source_file"
    )
)


display(
    episodios
    .filter(
        (F.col("series_id") == "lost")
        & (F.col("season") == "6")
        & F.expr(
            "try_cast(episode_num_overall AS INT) BETWEEN 103 AND 116"
        )
    )
    .select(
        "season",
        "episode_num_in_season",
        "episode_num_overall",
        "title",
        "original_air_date"
    )
    .orderBy(F.expr("try_cast(episode_num_overall AS INT)"))
)


display(
    imdb
    .filter(
        (F.col("series_id") == "lost")
        & (F.col("season") == "6")
        & (F.col("episode_num") == "12")
    )
    .select(
        "series_id",
        "season",
        "episode_num",
        "title",
        "original_air_date",
        "imdb_rating",
        "total_votes"
    )
)

series_id,season,episode_num_in_season,episode_num_overall,title,original_air_date,prod_code,us_viewers,_source_file
lost,6,12,105,LA X - Part 2,2010-02-02,null,12090000.0,dbfs:/Volumes/mvp_engdados/bronze/datafiles/lost_episodes.csv
lost,6,12,115,Everybody Loves Hugo,2010-04-13,null,9480000.0,dbfs:/Volumes/mvp_engdados/bronze/datafiles/lost_episodes.csv


season,episode_num_in_season,episode_num_overall,title,original_air_date
6,1,104,LA X - Part 1,2010-02-02
6,12,105,LA X - Part 2,2010-02-02
6,3,106,What Kate Does,2010-02-09
6,4,107,The Substitute,2010-02-16
6,5,108,Lighthouse,2010-02-23
6,6,109,Sundown,2010-03-02
6,7,110,Dr. Linus,2010-03-09
6,8,111,Recon,2010-03-16
6,9,112,Ab Aeterno,2010-03-23
6,10,113,The Package,2010-03-30


series_id,season,episode_num,title,original_air_date,imdb_rating,total_votes
lost,6,12,Everybody Loves Hugo,13 Apr. 2010,8.4,4164


### 3. Transformação para Silver

Na Silver, convertemos os campos para tipos adequados, padronizamos as datas e preservamos colunas de origem para rastrear as decisões. O campo de episódio pode conter referências entre colchetes, como 6[98]. Essa referência é removida antes da conversão porque não faz parte do número. 
É utilizado try_cast para que um valor ainda inválido resulte em nulo e possa ser auditado.
Também é aplicada a correção documentada de Lost. Ela não altera a Bronze e deixa uma coluna indicando que a correção foi aplicada.

In [0]:
imdb_raw = spark.table("mvp_engdados.bronze.imdb_raw")

imdb_silver = (
    imdb_raw
    .withColumn("season_raw", F.col("season"))
    .withColumn("episode_num_raw", F.col("episode_num"))
    .withColumn("imdb_rating_raw", F.col("imdb_rating"))
    .withColumn("total_votes_raw", F.col("total_votes"))
    .withColumnRenamed(
        "original_air_date",
        "original_air_date_raw"
    )
    .withColumn(
        "_date_normalized",
        F.regexp_replace(
            F.trim(F.col("original_air_date_raw")),
            r"\.",
            ""
        )
    )
    .withColumn(
        "season",
        F.expr("try_cast(nullif(trim(season_raw), '') AS INT)")
    )
    .withColumn(
        "episode_num",
        F.expr("try_cast(nullif(trim(episode_num_raw), '') AS INT)")
    )
    .withColumn(
        "imdb_rating",
        F.expr("try_cast(nullif(trim(imdb_rating_raw), '') AS DOUBLE)")
    )
    .withColumn(
        "total_votes",
        F.expr("try_cast(nullif(trim(total_votes_raw), '') AS BIGINT)")
    )
    .withColumn(
        "original_air_date",
        F.expr("""
            CAST(
                coalesce(
                    try_to_timestamp(_date_normalized, 'yyyy-MM-dd'),
                    try_to_timestamp(_date_normalized, 'd MMM yyyy')
                ) AS DATE
            )
        """)
    )
    .drop("_date_normalized")
)

# Confirme que o campo temporário foi removido
print("_date_normalized" in imdb_silver.columns)

(
    imdb_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_engdados.silver.imdb")
)

False


### 4. Validações da Silver

Depois da conversão, é verificado se a correção de Lost removeu a duplicidade candidata, se sobraram datas não convertidas e se existem valores fora dos domínios esperados. 
Rating IMDb deve estar entre 0 e 10. Votos e audiência não devem ser negativos. Resultado vazio nessas verificações significa que a regra consultada não encontrou ocorrências e não que a célula deixou de executar.

In [0]:
episodios = spark.table("mvp_engdados.silver.episodes")
imdb = spark.table("mvp_engdados.silver.imdb")

display(
    episodios
    .groupBy("series_id")
    .count()
    .orderBy("series_id")
)

display(
    imdb
    .groupBy("series_id")
    .count()
    .orderBy("series_id")
)


# Datas que não foram convertidas
display(
    episodios
    .filter(F.col("original_air_date").isNull())
    .select("series_id", "title", "original_air_date_raw")
)

display(
    imdb
    .filter(F.col("original_air_date").isNull())
    .select("series_id", "title", "original_air_date_raw")
)


# Duplicidades candidatas por chave natural
display(
    episodios
    .groupBy("series_id", "season", "episode_num_in_season")
    .count()
    .filter(F.col("count") > 1)
    .orderBy("series_id", "season", "episode_num_in_season")
)

display(
    imdb
    .groupBy("series_id", "season", "episode_num")
    .count()
    .filter(F.col("count") > 1)
    .orderBy("series_id", "season", "episode_num")
)


# Valores fora do domínio
display(
    imdb
    .filter(
        F.col("imdb_rating").isNotNull()
        & (
            (F.col("imdb_rating") < 0)
            | (F.col("imdb_rating") > 10)
        )
    )
)

display(
    imdb
    .filter(F.col("total_votes") < 0)
)

display(
    episodios
    .filter(F.col("us_viewers") < 0)
)


# Confirmar correção de Lost
display(
    episodios
    .filter(
        (F.col("series_id") == "lost")
        & (F.col("season") == 6)
        & (F.col("episode_num_overall").between(104, 106))
    )
    .select(
        "season",
        "episode_num_in_season_source",
        "episode_num_in_season",
        "episode_num_overall",
        "title",
        "correction_applied"
    )
    .orderBy("episode_num_overall")
)

series_id,count
big_bang_theory,279
breaking_bad,62
brooklyn_99,153
friends,236
game_of_thrones,73
how_i_met_your_mother,208
lost,121
the_office,201
the_walking_dead,177


series_id,count
big_bang_theory,280
breaking_bad,62
brooklyn_99,153
friends,235
game_of_thrones,73
how_i_met_your_mother,208
lost,119
the_office,188
the_walking_dead,177


series_id,title,original_air_date_raw


series_id,title,original_air_date_raw


series_id,season,episode_num_in_season,count


series_id,season,episode_num,count


season,episode_num,title,original_air_date_raw,imdb_rating,total_votes,desc,series_id,_source_file,_ingested_at,season_raw,episode_num_raw,imdb_rating_raw,total_votes_raw,original_air_date


season,episode_num,title,original_air_date_raw,imdb_rating,total_votes,desc,series_id,_source_file,_ingested_at,season_raw,episode_num_raw,imdb_rating_raw,total_votes_raw,original_air_date


season,episode_num_in_season,episode_num_overall,title,directed_by,written_by,original_air_date_raw,prod_code,us_viewers,series_id,_source_file,_ingested_at,featured_character,season_raw,episode_num_in_season_source,episode_num_overall_raw,us_viewers_raw,episode_num_in_season_clean,correction_applied,original_air_date


season,episode_num_in_season_source,episode_num_in_season,episode_num_overall,title,correction_applied
6,1,1,104,LA X - Part 1,false
6,12,2,105,LA X - Part 2,true
6,3,3,106,What Kate Does,false


### 5. Reconciliação e pareamento das fontes

As fontes não têm sempre a mesma numeração ou o mesmo conjunto de episódios. Primeiro, fazemos uma reconciliação pela série, temporada e número do episódio e classificamos o resultado para investigar divergências.
Para análises que precisam combinar audiência e nota, usamos um critério mais restritivo: série, título normalizado e data, com a chave aparecendo uma única vez em cada fonte. 
Isso evita associações falsas por número de episódio. Registros sem correspondência segura continuam preservados nas tabelas de origem e ficam fora da tabela pareada.

In [0]:
episodios = spark.table("mvp_engdados.silver.episodes")
imdb = spark.table("mvp_engdados.silver.imdb")


# Reconciliação inicial pela numeração de temporada/episódio
e = (
    episodios
    .select(
        "series_id",
        "season",
        F.col("episode_num_in_season").alias("episode_number"),
        F.col("title").alias("episode_title"),
        F.col("original_air_date").alias("episode_date"),
        F.col("episode_num_overall"),
        F.lit(True).alias("_episode_exists")
    )
    .alias("e")
)

i = (
    imdb
    .select(
        "series_id",
        "season",
        F.col("episode_num").alias("episode_number"),
        F.col("title").alias("imdb_title"),
        F.col("original_air_date").alias("imdb_date"),
        "imdb_rating",
        "total_votes",
        F.lit(True).alias("_imdb_exists")
    )
    .alias("i")
)

condicao_numero = (
    (F.col("e.series_id") == F.col("i.series_id"))
    & (F.col("e.season") == F.col("i.season"))
    & (F.col("e.episode_number") == F.col("i.episode_number"))
)

reconciliacao = (
    e.join(i, condicao_numero, "full_outer")
    .select(
        F.coalesce(
            F.col("e.series_id"),
            F.col("i.series_id")
        ).alias("series_id"),
        F.coalesce(
            F.col("e.season"),
            F.col("i.season")
        ).alias("season"),
        F.coalesce(
            F.col("e.episode_number"),
            F.col("i.episode_number")
        ).alias("episode_num_candidate"),
        F.col("e._episode_exists"),
        F.col("i._imdb_exists"),
        F.col("e.episode_title"),
        F.col("i.imdb_title"),
        F.col("e.episode_date"),
        F.col("i.imdb_date"),
        F.col("e.episode_num_overall"),
        F.col("i.imdb_rating"),
        F.col("i.total_votes")
    )
    .withColumn(
        "title_match",
        F.lower(F.trim("episode_title"))
        == F.lower(F.trim("imdb_title"))
    )
    .withColumn(
        "date_match",
        F.col("episode_date") == F.col("imdb_date")
    )
    .withColumn(
        "match_status",
        F.when(
            F.col("_episode_exists").isNull(),
            "somente_imdb"
        )
        .when(
            F.col("_imdb_exists").isNull(),
            "somente_episodes"
        )
        .when(
            F.col("title_match") & F.col("date_match"),
            "confirmado"
        )
        .when(
            F.col("title_match"),
            "titulo_igual_data_diferente"
        )
        .when(
            F.col("date_match"),
            "data_igual_titulo_diferente"
        )
        .otherwise("titulo_e_data_diferentes")
    )
)

display(
    reconciliacao
    .groupBy("series_id", "match_status")
    .count()
    .orderBy("series_id", "match_status")
)

display(
    reconciliacao
    .filter(F.col("match_status") != "confirmado")
    .orderBy("series_id", "season", "episode_num_candidate")
)


# Chave alternativa: série, título padronizado e data
chaves = ["series_id", "_title_key", "_airdate"]

e = (
    episodios
    .filter(
        F.col("title").isNotNull()
        & F.col("original_air_date").isNotNull()
    )
    .withColumn(
        "_title_key",
        F.lower(
            F.regexp_replace(
                F.trim("title"),
                r"\s+",
                " "
            )
        )
    )
    .withColumn("_airdate", F.col("original_air_date"))
)

i = (
    imdb
    .filter(
        F.col("title").isNotNull()
        & F.col("original_air_date").isNotNull()
    )
    .withColumn(
        "_title_key",
        F.lower(
            F.regexp_replace(
                F.trim("title"),
                r"\s+",
                " "
            )
        )
    )
    .withColumn("_airdate", F.col("original_air_date"))
)


def manter_chaves_unicas(df, chaves):
    unicas = (
        df.groupBy(*chaves)
        .count()
        .filter(F.col("count") == 1)
        .select(*chaves)
    )

    return df.join(unicas, on=chaves, how="inner")


e_unicos = manter_chaves_unicas(e, chaves)
i_unicos = manter_chaves_unicas(i, chaves)

pares_seguros = (
    e_unicos.alias("e")
    .join(i_unicos.alias("i"), on=chaves, how="inner")
    .select(
        F.col("e.series_id").alias("series_id"),
        F.col("e.season").alias("episode_season"),
        F.col("e.episode_num_in_season").alias("episode_number"),
        F.col("i.season").alias("imdb_season"),
        F.col("i.episode_num").alias("imdb_episode_number"),
        F.col("e.title").alias("episode_title"),
        F.col("e.original_air_date").alias("air_date"),
        F.col("e.us_viewers").alias("us_viewers"),
        F.col("i.imdb_rating").alias("imdb_rating"),
        F.col("i.total_votes").alias("total_votes")
    )
    .withColumn(
        "match_method",
        F.lit(
            "series_id + título normalizado + data; "
            "chave única nas duas fontes"
        )
    )
)

display(
    pares_seguros
    .groupBy("series_id")
    .count()
    .orderBy("series_id")
)

display(
    pares_seguros
    .filter(
        (F.col("episode_season") != F.col("imdb_season"))
        | (F.col("episode_number") != F.col("imdb_episode_number"))
    )
    .orderBy("series_id", "episode_season", "episode_number")
)


(
    pares_seguros.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_engdados.silver.episodes_imdb_matched")
)

series_id,match_status,count
big_bang_theory,confirmado,262
big_bang_theory,somente_imdb,1
big_bang_theory,titulo_e_data_diferentes,17
breaking_bad,confirmado,62
brooklyn_99,confirmado,153
friends,confirmado,235
friends,somente_episodes,1
game_of_thrones,confirmado,73
how_i_met_your_mother,confirmado,208
lost,confirmado,119


series_id,season,episode_num_candidate,_episode_exists,_imdb_exists,episode_title,imdb_title,episode_date,imdb_date,episode_num_overall,imdb_rating,total_votes,title_match,date_match,match_status
big_bang_theory,1,1,true,true,Pilot,Unaired Pilot,2007-09-24,2006-05-01,1,6.7,2048,false,false,titulo_e_data_diferentes
big_bang_theory,1,2,true,true,The Big Bran Hypothesis,Pilot,2007-10-01,2007-09-24,2,8.2,6135,false,false,titulo_e_data_diferentes
big_bang_theory,1,3,true,true,The Fuzzy Boots Corollary,The Big Bran Hypothesis,2007-10-08,2007-10-01,3,8.3,4924,false,false,titulo_e_data_diferentes
big_bang_theory,1,4,true,true,The Luminous Fish Effect,The Fuzzy Boots Corollary,2007-10-15,2007-10-08,4,7.7,4392,false,false,titulo_e_data_diferentes
big_bang_theory,1,5,true,true,The Hamburger Postulate,The Luminous Fish Effect,2007-10-22,2007-10-15,5,8.1,4434,false,false,titulo_e_data_diferentes
big_bang_theory,1,6,true,true,The Middle Earth Paradigm,The Hamburger Postulate,2007-10-29,2007-10-22,6,7.9,4227,false,false,titulo_e_data_diferentes
big_bang_theory,1,7,true,true,The Dumpling Paradox,The Middle Earth Paradigm,2007-11-05,2007-10-29,7,8.4,4402,false,false,titulo_e_data_diferentes
big_bang_theory,1,8,true,true,The Grasshopper Experiment,The Dumpling Paradox,2007-11-12,2007-11-05,8,8.1,4143,false,false,titulo_e_data_diferentes
big_bang_theory,1,9,true,true,The Cooper-Hofstadter Polarization,The Grasshopper Experiment,2008-03-17,2007-11-12,9,8.2,4172,false,false,titulo_e_data_diferentes
big_bang_theory,1,10,true,true,The Loobenfeld Decay,The Cooper-Hofstadter Polarization,2008-03-24,2008-03-17,10,8.0,4024,false,false,titulo_e_data_diferentes


series_id,count
big_bang_theory,279
breaking_bad,62
brooklyn_99,153
friends,234
game_of_thrones,73
how_i_met_your_mother,208
lost,117
the_office,175
the_walking_dead,177


series_id,episode_season,episode_number,imdb_season,imdb_episode_number,episode_title,air_date,us_viewers,imdb_rating,total_votes,match_method
big_bang_theory,1,1,1,2,Pilot,2007-09-24,9520000.0,8.2,6135,series_id + título normalizado + data; chave única nas duas fontes
big_bang_theory,1,2,1,3,The Big Bran Hypothesis,2007-10-01,8580000.0,8.3,4924,series_id + título normalizado + data; chave única nas duas fontes
big_bang_theory,1,3,1,4,The Fuzzy Boots Corollary,2007-10-08,8359999.999999999,7.7,4392,series_id + título normalizado + data; chave única nas duas fontes
big_bang_theory,1,4,1,5,The Luminous Fish Effect,2007-10-15,8150000.0,8.1,4434,series_id + título normalizado + data; chave única nas duas fontes
big_bang_theory,1,5,1,6,The Hamburger Postulate,2007-10-22,8810000.0,7.9,4227,series_id + título normalizado + data; chave única nas duas fontes
big_bang_theory,1,6,1,7,The Middle Earth Paradigm,2007-10-29,8920000.0,8.4,4402,series_id + título normalizado + data; chave única nas duas fontes
big_bang_theory,1,7,1,8,The Dumpling Paradox,2007-11-05,9680000.0,8.1,4143,series_id + título normalizado + data; chave única nas duas fontes
big_bang_theory,1,8,1,9,The Grasshopper Experiment,2007-11-12,9320000.0,8.2,4172,series_id + título normalizado + data; chave única nas duas fontes
big_bang_theory,1,9,1,10,The Cooper-Hofstadter Polarization,2008-03-17,9110000.0,8.0,4024,series_id + título normalizado + data; chave única nas duas fontes
big_bang_theory,1,10,1,11,The Loobenfeld Decay,2008-03-24,8630000.0,8.0,4119,series_id + título normalizado + data; chave única nas duas fontes


### 6. Modelagem das tabelas Gold

A tabela series_metrics tem uma linha por série. A tabela season_metrics tem uma linha por série e temporada. As contagens e médias de audiência vêm da fonte de episódios, as notas vêm da fonte IMDb e são agregadas separadamente antes da junção. Assim, diferenças de quantidade entre as fontes não multiplicam registros nem distorcem as médias.
O episódio “Unaired Pilot” de The Big Bang Theory é excluído da análise de notas, pois não é um episódio exibido. A tabela de correlação terá uma linha por série e será calculada somente a partir dos episódios pareados com segurança e com audiência e nota disponíveis.

In [0]:
episodios = spark.table("mvp_engdados.silver.episodes")
imdb = spark.table("mvp_engdados.silver.imdb")


# Excluir o piloto não exibido da análise de notas
imdb_para_analise = imdb.filter(
    ~(
        (F.col("series_id") == "big_bang_theory")
        & (F.lower(F.trim("title")) == "unaired pilot")
    )
)


# Uma linha por série
episodios_por_serie = (
    episodios
    .groupBy("series_id")
    .agg(
        F.countDistinct("episode_num_overall")
        .alias("episode_count"),
        F.countDistinct("season")
        .alias("season_count"),
        F.min("original_air_date")
        .alias("first_air_date"),
        F.max("original_air_date")
        .alias("last_air_date"),
        F.count("us_viewers")
        .alias("audience_episode_count"),
        F.round(F.avg("us_viewers"), 0)
        .alias("avg_us_viewers")
    )
    .withColumn(
        "duration_days",
        F.datediff("last_air_date", "first_air_date")
    )
    .withColumn(
        "duration_years",
        F.round(F.col("duration_days") / F.lit(365.25), 2)
    )
)

notas_por_serie = (
    imdb_para_analise
    .groupBy("series_id")
    .agg(
        F.count("imdb_rating").alias("imdb_episode_count"),
        F.round(F.avg("imdb_rating"), 2)
        .alias("avg_imdb_rating")
    )
)

gold_series_metrics = episodios_por_serie.join(
    notas_por_serie,
    on="series_id",
    how="full_outer"
)


# Uma linha por série e temporada
audiencia_por_temporada = (
    episodios
    .groupBy("series_id", "season")
    .agg(
        F.countDistinct("episode_num_overall")
        .alias("episode_count"),
        F.count("us_viewers")
        .alias("audience_episode_count"),
        F.round(F.avg("us_viewers"), 0)
        .alias("avg_us_viewers")
    )
)

notas_por_temporada = (
    imdb_para_analise
    .groupBy("series_id", "season")
    .agg(
        F.count("imdb_rating").alias("imdb_episode_count"),
        F.round(F.avg("imdb_rating"), 2)
        .alias("avg_imdb_rating")
    )
)

gold_season_metrics = audiencia_por_temporada.join(
    notas_por_temporada,
    on=["series_id", "season"],
    how="full_outer"
)


# Gravar as tabelas
(
    gold_series_metrics.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_engdados.gold.series_metrics")
)

(
    gold_season_metrics.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_engdados.gold.season_metrics")
)


display(
    spark.table("mvp_engdados.gold.series_metrics")
    .orderBy("series_id")
)

display(
    spark.table("mvp_engdados.gold.season_metrics")
    .orderBy("series_id", "season")
)

series_id,episode_count,season_count,first_air_date,last_air_date,audience_episode_count,avg_us_viewers,duration_days,duration_years,imdb_episode_count,avg_imdb_rating
big_bang_theory,279,12,2007-09-24,2019-05-16,279,1.4646308E7,4252,11.64,279,7.82
breaking_bad,62,5,2008-01-20,2013-09-29,57,2324386.0,2079,5.69,62,9.03
brooklyn_99,153,8,2013-09-17,2021-09-16,153,2620719.0,2921,8.0,153,8.13
friends,236,10,1994-09-22,2004-05-06,236,2.5390805E7,3514,9.62,235,8.42
game_of_thrones,73,8,2011-04-17,2019-05-19,73,6447808.0,2954,8.09,73,8.75
how_i_met_your_mother,208,9,2005-09-19,2014-03-31,208,9044183.0,3115,8.53,208,8.14
lost,121,6,2004-09-22,2010-05-23,121,1.4580992E7,2069,5.66,119,8.55
the_office,201,9,2005-03-24,2013-05-16,201,7358856.0,2975,8.15,188,8.22
the_walking_dead,177,11,2010-10-31,2022-11-20,177,8137966.0,4403,12.05,177,7.92


series_id,season,episode_count,audience_episode_count,avg_us_viewers,imdb_episode_count,avg_imdb_rating
big_bang_theory,1,17,17,8443529.0,17,8.14
big_bang_theory,2,23,23,1.0142609E7,23,8.24
big_bang_theory,3,23,23,1.4118696E7,23,8.26
big_bang_theory,4,24,24,1.2558333E7,24,8.11
big_bang_theory,5,24,24,1.488125E7,24,8.0
big_bang_theory,6,24,24,1.6799167E7,24,8.0
big_bang_theory,7,24,24,1.7695E7,24,7.9
big_bang_theory,8,24,24,1.74975E7,24,7.49
big_bang_theory,9,24,24,1.770875E7,24,7.58
big_bang_theory,10,24,24,1.6670833E7,24,7.41


### 7. Correlação audiência × Nota IMDb

A correlação de Pearson mede a associação linear entre duas variáveis numéricas. Aqui ela é calculada por série, usando somente episódios pareados com audiência e nota preenchidas. O resultado é descritivo: correlação não demonstra que audiência causa uma nota maior ou menor. A coluna episode_count informa o tamanho da amostra usada para cada série.

In [0]:
pares = spark.table("mvp_engdados.silver.episodes_imdb_matched")

pares_validos = pares.filter(
    F.col("us_viewers").isNotNull()
    & F.col("imdb_rating").isNotNull()
)

correlacao_por_serie = (
    pares_validos
    .groupBy("series_id")
    .agg(
        F.count("*").alias("episode_count"),
        F.round(
            F.corr("us_viewers", "imdb_rating"),
            3
        ).alias("pearson_correlation"),
        F.round(F.avg("us_viewers"), 0)
        .alias("avg_us_viewers"),
        F.round(F.avg("imdb_rating"), 2)
        .alias("avg_imdb_rating")
    )
)

(
    correlacao_por_serie.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_engdados.gold.audience_rating_correlation")
)

display(
    spark.table("mvp_engdados.gold.audience_rating_correlation")
    .orderBy("pearson_correlation")
)

series_id,episode_count,pearson_correlation,avg_us_viewers,avg_imdb_rating
game_of_thrones,73,-0.431,6447808.0,8.75
big_bang_theory,279,-0.193,1.4646308E7,7.82
lost,117,-0.002,1.4537607E7,8.53
brooklyn_99,153,0.007,2620719.0,8.13
how_i_met_your_mother,208,0.11,9044183.0,8.14
friends,234,0.276,2.5159444E7,8.42
the_walking_dead,177,0.28,8137966.0,7.92
the_office,175,0.511,7117086.0,8.16
breaking_bad,57,0.54,2324386.0,9.04


### 8. Análise das perguntas principais

As consultas abaixo preparam as tabelas que respondem às perguntas de negócio.

In [0]:
series_gold = spark.table("mvp_engdados.gold.series_metrics")
temporadas_gold = spark.table("mvp_engdados.gold.season_metrics")
correlacao_gold = spark.table(
    "mvp_engdados.gold.audience_rating_correlation"
)


# Pergunta 1: qual série teve maior intervalo de exibição?
display(
    series_gold
    .select(
        "series_id",
        "first_air_date",
        "last_air_date",
        "duration_days",
        "duration_years"
    )
    .orderBy(F.desc("duration_days"))
)


# Pergunta 2: qual série possui mais episódios?
display(
    series_gold
    .select("series_id", "episode_count")
    .orderBy(F.desc("episode_count"))
)


# Pergunta 3: qual série tem maior nota média no IMDb?
display(
    series_gold
    .select(
        "series_id",
        "avg_imdb_rating",
        "imdb_episode_count"
    )
    .orderBy(F.desc("avg_imdb_rating"))
)


# Pergunta 4: como a nota média varia por temporada?
display(
    temporadas_gold
    .select(
        "series_id",
        "season",
        "imdb_episode_count",
        "avg_imdb_rating"
    )
    .orderBy("series_id", "season")
)


# Pergunta 5: como a audiência média varia por temporada?
display(
    temporadas_gold
    .select(
        "series_id",
        "season",
        "audience_episode_count",
        "avg_us_viewers"
    )
    .orderBy("series_id", "season")
)


# Pergunta 6: qual a correlação entre audiência e avaliação por série?
display(
    correlacao_gold
    .select(
        "series_id",
        "episode_count",
        "pearson_correlation"
    )
    .orderBy("pearson_correlation")
)

series_id,first_air_date,last_air_date,duration_days,duration_years
the_walking_dead,2010-10-31,2022-11-20,4403,12.05
big_bang_theory,2007-09-24,2019-05-16,4252,11.64
friends,1994-09-22,2004-05-06,3514,9.62
how_i_met_your_mother,2005-09-19,2014-03-31,3115,8.53
the_office,2005-03-24,2013-05-16,2975,8.15
game_of_thrones,2011-04-17,2019-05-19,2954,8.09
brooklyn_99,2013-09-17,2021-09-16,2921,8.0
breaking_bad,2008-01-20,2013-09-29,2079,5.69
lost,2004-09-22,2010-05-23,2069,5.66


Databricks visualization. Run in Databricks to view.

series_id,episode_count
big_bang_theory,279
friends,236
how_i_met_your_mother,208
the_office,201
the_walking_dead,177
brooklyn_99,153
lost,121
game_of_thrones,73
breaking_bad,62


Databricks visualization. Run in Databricks to view.

series_id,avg_imdb_rating,imdb_episode_count
breaking_bad,9.03,62
game_of_thrones,8.75,73
lost,8.55,119
friends,8.42,235
the_office,8.22,188
how_i_met_your_mother,8.14,208
brooklyn_99,8.13,153
the_walking_dead,7.92,177
big_bang_theory,7.82,279


...

Databricks visualization. Run in Databricks to view.

series_id,season,audience_episode_count,avg_us_viewers
big_bang_theory,1,17,8443529.0
big_bang_theory,2,23,1.0142609E7
big_bang_theory,3,23,1.4118696E7
big_bang_theory,4,24,1.2558333E7
big_bang_theory,5,24,1.488125E7
big_bang_theory,6,24,1.6799167E7
big_bang_theory,7,24,1.7695E7
big_bang_theory,8,24,1.74975E7
big_bang_theory,9,24,1.770875E7
big_bang_theory,10,24,1.6670833E7


Databricks visualization. Run in Databricks to view.

series_id,episode_count,pearson_correlation
game_of_thrones,73,-0.431
big_bang_theory,279,-0.193
lost,117,-0.002
brooklyn_99,153,0.007
how_i_met_your_mother,208,0.11
friends,234,0.276
the_walking_dead,177,0.28
the_office,175,0.511
breaking_bad,57,0.54


Databricks visualization. Run in Databricks to view.

In [0]:
pares = spark.table("mvp_engdados.silver.episodes_imdb_matched")

display(
    pares
    .groupBy("series_id")
    .agg(
        F.count("*").alias("episodios_pareados"),
        F.count("us_viewers").alias("com_audiencia"),
        F.count("imdb_rating").alias("com_nota_imdb"),
        F.sum(
            F.when(
                F.col("us_viewers").isNotNull()
                & F.col("imdb_rating").isNotNull(),
                1
            ).otherwise(0)
        ).alias("validos_para_correlacao")
    )
    .orderBy("series_id")
)

series_id,episodios_pareados,com_audiencia,com_nota_imdb,validos_para_correlacao
big_bang_theory,279,279,279,279
breaking_bad,62,57,62,57
brooklyn_99,153,153,153,153
friends,234,234,234,234
game_of_thrones,73,73,73,73
how_i_met_your_mother,208,208,208,208
lost,117,117,117,117
the_office,175,175,175,175
the_walking_dead,177,177,177,177


### 9. Análises complementares

Além das seis perguntas principais, estas duas análises enriquecem a interpretação. A primeira calcula a variação percentual de audiência de uma temporada para a seguinte. A segunda mede a dispersão das notas IMDb dentro de cada série: um desvio padrão menor indica notas mais concentradas e um maior indica maior variação entre episódios.

In [0]:
temporadas = spark.table("mvp_engdados.gold.season_metrics")

janela = (
    Window
    .partitionBy("series_id")
    .orderBy("season")
)

variacao_audiencia = (
    temporadas
    .select("series_id", "season", "avg_us_viewers")
    .withColumn(
        "previous_season_viewers",
        F.lag("avg_us_viewers").over(janela)
    )
    .withColumn(
        "audience_change_pct",
        F.when(
            F.col("previous_season_viewers").isNotNull()
            & (F.col("previous_season_viewers") != 0),
            F.round(
                (
                    (
                        F.col("avg_us_viewers")
                        - F.col("previous_season_viewers")
                    )
                    / F.col("previous_season_viewers")
                ) * 100,
                2
            )
        )
    )
)

(
    variacao_audiencia.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_engdados.gold.audience_season_change")
)

display(
    spark.table("mvp_engdados.gold.audience_season_change")
    .filter(F.col("audience_change_pct").isNotNull())
    .orderBy(F.desc(F.abs("audience_change_pct")))
)


# Dispersão das notas por série
imdb = spark.table("mvp_engdados.silver.imdb")

imdb_para_analise = imdb.filter(
    ~(
        (F.col("series_id") == "big_bang_theory")
        & (F.lower(F.trim("title")) == "unaired pilot")
    )
)

consistencia_notas = (
    imdb_para_analise
    .groupBy("series_id")
    .agg(
        F.count("imdb_rating").alias("imdb_episode_count"),
        F.round(F.avg("imdb_rating"), 2)
        .alias("avg_imdb_rating"),
        F.round(F.stddev_samp("imdb_rating"), 2)
        .alias("rating_stddev"),
        F.round(
            F.percentile_approx("imdb_rating", 0.5),
            2
        ).alias("median_imdb_rating"),
        F.min("imdb_rating").alias("min_imdb_rating"),
        F.max("imdb_rating").alias("max_imdb_rating")
    )
)

(
    consistencia_notas.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_engdados.gold.rating_consistency")
)

display(
    spark.table("mvp_engdados.gold.rating_consistency")
    .orderBy("rating_stddev")
)

series_id,season,avg_us_viewers,previous_season_viewers,audience_change_pct
breaking_bad,5,4315625.0,1866923.0,131.16
the_walking_dead,3,1.075125E7,6911538.0,55.56
game_of_thrones,2,3795000.0,2515000.0,50.89
the_walking_dead,11,1690417.0,3036818.0,-44.34
big_bang_theory,3,1.4118696E7,1.0142609E7,39.2
the_walking_dead,10,3036818.0,4951875.0,-38.67
game_of_thrones,4,6846000.0,4966000.0,37.86
the_walking_dead,9,4951875.0,7817500.0,-36.66
game_of_thrones,7,1.0261429E7,7688000.0,33.47
the_walking_dead,2,6911538.0,5235000.0,32.03


Databricks visualization. Run in Databricks to view.

series_id,imdb_episode_count,avg_imdb_rating,rating_stddev,median_imdb_rating,min_imdb_rating,max_imdb_rating
friends,235,8.42,0.41,8.4,7.1,9.7
lost,119,8.55,0.47,8.5,7.2,9.7
big_bang_theory,279,7.82,0.5,7.8,6.8,9.5
breaking_bad,62,9.03,0.51,8.9,7.9,10.0
how_i_met_your_mother,208,8.14,0.58,8.1,5.5,9.5
brooklyn_99,153,8.13,0.58,8.1,4.8,9.6
the_office,188,8.22,0.62,8.2,6.4,9.8
the_walking_dead,177,7.92,0.82,7.9,4.1,9.6
game_of_thrones,73,8.75,0.93,8.8,4.1,9.9


Databricks visualization. Run in Databricks to view.

### 10. Catálogo e documentação das tabelas

As tabelas Gold têm grãos diferentes: uma linha por série em series_metrics. Uma linha por série e temporada em season_metrics e uma linha por série em cada tabela de correlação e consistência. A tabela pareada da Silver contém episódios encontrados de forma segura nas duas fontes. Essa descrição informa o que cada linha representa e como interpretar as relações.

In [0]:
comentarios_tabelas = {
    "mvp_engdados.bronze.episodes_raw":
        "Dados brutos de episódios, lidos dos CSVs sem tipagem de negócio.",
    "mvp_engdados.bronze.imdb_raw":
        "Dados brutos de episódios e avaliações IMDb, lidos dos CSVs.",
    "mvp_engdados.silver.episodes":
        "Episódios tipados e padronizados; preserva campos de origem e correção aplicada.",
    "mvp_engdados.silver.imdb":
        "Dados IMDb tipados; preserva campos de origem e datas originais.",
    "mvp_engdados.silver.episodes_imdb_matched":
        "Episódios pareados entre fontes por série, título normalizado e data, com chave única.",
    "mvp_engdados.gold.series_metrics":
        "Métricas agregadas com uma linha por série.",
    "mvp_engdados.gold.season_metrics":
        "Métricas agregadas com uma linha por série e temporada.",
    "mvp_engdados.gold.audience_rating_correlation":
        "Correlação de Pearson por série calculada sobre episódios pareados válidos.",
    "mvp_engdados.gold.audience_season_change":
        "Variação percentual de audiência média entre temporadas consecutivas.",
    "mvp_engdados.gold.rating_consistency":
        "Estatísticas de tendência central e dispersão das notas IMDb por série."
}

for tabela, comentario in comentarios_tabelas.items():
    comentario_sql = comentario.replace("'", "''")
    spark.sql(
        f"COMMENT ON TABLE {tabela} IS '{comentario_sql}'"
    )


display(
    spark.sql("""
        SELECT
            table_schema,
            table_name,
            column_name,
            ordinal_position,
            data_type,
            comment
        FROM mvp_engdados.information_schema.columns
        WHERE table_schema IN ('bronze', 'silver', 'gold')
        ORDER BY table_schema, table_name, ordinal_position
    """)
)

table_schema,table_name,column_name,ordinal_position,data_type,comment
bronze,episodes_raw,season,0,STRING,null
bronze,episodes_raw,episode_num_in_season,1,STRING,null
bronze,episodes_raw,episode_num_overall,2,STRING,null
bronze,episodes_raw,title,3,STRING,null
bronze,episodes_raw,directed_by,4,STRING,null
bronze,episodes_raw,written_by,5,STRING,null
bronze,episodes_raw,original_air_date,6,STRING,null
bronze,episodes_raw,prod_code,7,STRING,null
bronze,episodes_raw,us_viewers,8,STRING,null
bronze,episodes_raw,series_id,9,STRING,null
